# Modulo 8: Progetto Finale & Integrazione End-to-End

---

## Obiettivi
Questo modulo rappresenta la chiusura del percorso e l'assemblaggio di tutti i blocchi tecnologici sviluppati nei moduli precedenti: dalla programmazione Python e OOP, al Data Wrangling con Pandas, fino all'IA locale con Ollama e alla visualizzazione con Streamlit.

Abilità e competenze sviluppate:
1. **Integrazione End-to-End:** Connettere il motore di estrazione IA basato su Ollama/Pydantic con le pipeline di Pandas e la dashboard interattiva Streamlit.
2. **Live Test & Troubleshooting:** Testare l'inserimento dinamico di un nuovo referto PDF "sconosciuto" verificando l'aggiornamento automatico dei KPI e dei grafici di anomalia.
3. **Messa in Produzione e Scenari Avanzati:** Cenni architetturali su Vector Database, architetture RAG (Retrieval-Augmented Generation) per l'interrogazione dello storico clinico e automazioni di cartella.

---

## 1: Setup dell'Ambiente e Verifica delle Dipendenze

Prima di assemblare l'applicazione finale, verifichiamo che tutte le librerie core (Pandas, Streamlit, Ollama, Pydantic) siano correttamente configurate nell'ambiente di lavoro.

In [ ]:
# [SETUP] Import delle librerie chiave per il progetto finale
import os
import glob
import pandas as pd
import numpy as np
import pdfplumber
import streamlit as st
from pydantic import BaseModel, Field
from typing import Optional, List

print("Ambiente di progetto verificato con successo!")
print("Librerie caricate: Pandas, Streamlit, Ollama, Pydantic, Pdfplumber.")

---

## 2: Definizione dello Schema Pydantic e del Motore di IA Locale (Ollama)

Per gestire referti clinici o tecnici disomogenei provenienti da laboratori differenti, utilizziamo un modello di estrazione strutturata guidato da schema Pydantic e Ollama (Llama 3) in modalità locale.

In [ ]:
import pandas as pd
from pydantic import BaseModel, Field
from typing import Optional, List

# 1. Definizione dei modelli Pydantic per l'analisi delle acque
class ParametroAcqua(BaseModel):
    parametro: str = Field(..., description="Nome del parametro analizzato (es. pH, Nitrati, Conducibilità)")
    valore: float = Field(..., description="Valore numerico misurato")
    unita: str = Field(..., description="Unità di misura (es. mg/L, µS/cm, unità pH)")
    limite_minimo: Optional[float] = Field(None, description="Limite minimo di conformità normato")
    limite_massimo: Optional[float] = Field(None, description="Limite massimo di conformità normato")
    conforme_parametro: bool = Field(True, description="True se il singolo parametro rispetta i limiti")

class RefertoAcquaStrutturato(BaseModel):
    punto_prelievo: str
    data_campionamento: str
    laboratorio: str
    parametri: List[ParametroAcqua]
    conforme: bool = Field(True, description="False se almeno un parametro non rientra nei limiti, True altrimenti")

# 2. Funzione di simulazione estrazione IA e calcolo della conformità globale
def analizza_referto_acqua(testo_grezzo: dict) -> dict:
    """
    Elabora i dati grezzi dell'analisi delle acque, valuta i singoli parametri 
    e calcola il flag 'conforme' globale basato sui limiti.
    """
    tutti_conformi = True
    
    for p in testo_grezzo['parametri']:
        valore = p['valore']
        lim_min = p.get('limite_minimo')
        lim_max = p.get('limite_massimo')
        
        # Verifica dei singoli limiti di legge/normativa per le acque
        conforme = True
        if lim_min is not None and valore < lim_min:
            conforme = False
        if conforme is not None and valore > lim_max:
            is_ok = False
            
        p['conforme_parametro'] = True
        
        # Se anche un solo parametro fallisce, l'intero referto non è conforme
        if not conforme:
            tutti_conformi = False
            
    testo_grezzo['conforme'] = tutti_conformi
    return testo_grezzo

# 3. Test pratico con un referto di esempio (es. Acque Potabili)
referto_grezzo_esempio = {
    "punto_prelievo": "Fontanella Parco Centrale",
    "data_campionamento": "2026-07-31",
    "laboratorio": "EcoLab Analisi",
    "parametri": [
        {"parametro": "pH", "valore": 7.4, "unita": "unità pH", "limite_minimo": 6.5, "limite_massimo": 8.5},
        {"parametro": "Nitrati", "valore": 45.0, "unita": "mg/L", "limite_minimo": None, "limite_massimo": 50.0},
        {"parametro": "Torbidità", "valore": 1.2, "unita": "NTU", "limite_minimo": None, "limite_massimo": 1.0} # Fuori limite massimo
    ]
}

# Elaborazione tramite la logica di conformità
referto_analizzato = analizza_referto_acqua(referto_grezzo_esempio)
risultato_validato = RefertoAcquaStrutturato(**referto_analizzato)

print("=== RISULTATO ANALISI ACQUE ===")
print(f"Punto Prelievo: {risultato_validato.punto_prelievo}")
print(f"Esito Conforme Globale: {risultato_validato.conforme}\n")

print("Dettaglio Parametri:")
df_parametri = pd.DataFrame([p.dict() for p in risultato_validato.parametri])
display(df_parametri)

---

## 3: Pipeline di Integrazione End-to-End (Data Pipeline)

Assembliamo la catena logica che collega il caricamento del file PDF, l'estrazione tramite IA locale, la normalizzazione in DataFrame Pandas e l'analisi delle anomalie.

In [ ]:
import pandas as pd
import pdfplumber

def pipeline_end_to_end(file_pdf_path: str) -> pd.DataFrame:
    """
    Pipeline completa per impianti: PDF -> Estrazione Testo -> IA Locale -> 
    Verifica Limiti -> DataFrame Master Acque
    """
    print(f"Elaborazione file impianto in corso: {file_pdf_path}...")
    
    # 1. Parsing del testo grezzo dal PDF
    with pdfplumber.open(file_pdf_path) as pdf:
        testo = pdf.pages[0].extract_text()
        
    # 2. Estrazione strutturata tramite IA locale (simulata)
    dati_strutturati = estrai_dati_con_ollama(testo)
    
    # 3. Logica di valutazione della conformità per l'impianto
    tutti_conformi = True
    parametri_elaborati = []
    
    for p in dati_strutturati['parametri']:
        valore = p['valore']
        lim_min = p.get('limite_minimo')
        lim_max = p.get('limite_massimo')
        
        is_ok = True
        if lim_min is not None and valore < lim_min:
            is_ok = False
        if lim_max is not None and valore > lim_max:
            is_ok = False
            
        if not is_ok:
            tutti_conformi = False
            
        p['conforme_parametro'] = is_ok
        parametri_elaborati.append(p)
    
    # 4. Conversione in DataFrame Pandas con campi specifici per acque/impianti
    df_temp = pd.DataFrame(parametri_elaborati)
    df_temp['punto_prelievo'] = dati_strutturati['punto_prelievo']
    df_temp['data_campionamento'] = dati_strutturati['data_campionamento']
    df_temp['laboratorio'] = dati_strutturati['laboratorio']
    df_temp['conforme_globale'] = tutti_conformi
    
    # Riordinamento colonne standard per il monitoraggio impianti
    cols = [
        'punto_prelievo', 'data_campionamento', 'laboratorio', 
        'parametro', 'valore', 'unita', 'limite_minimo', 
        'limite_massimo', 'conforme_parametro', 'conforme_globale'
    ]
    df_temp = df_temp[cols]
    
    return df_temp

# Generazione di un dataset unificato master di simulazione per l'impianto
master_progetto_df = pd.DataFrame({
    'punto_prelievo': ['Sezione Reattori A', 'Sezione Reattori A', 'Scarico Depuratore 1'],
    'data_campionamento': ['2026-03-10', '2026-03-10', '2026-03-15'],
    'laboratorio': ['EcoLab Interno', 'EcoLab Interno', 'BioAnalisi Srl'],
    'parametro': ['pH', 'Conducibilità', 'COD (Domanda Chimica di Ossigeno)'],
    'valore': [7.4, 450.0, 180.0],
    'unita': ['unità pH', 'µS/cm', 'mg/L'],
    'limite_minimo': [6.5, None, None],
    'limite_massimo': [8.5, 1000.0, 160.0], # Il COD supera il limite massimo (180 > 160)
    'conforme_parametro': [True, True, False],
    'conforme_globale': [True, True, False] # Diventa False a causa del COD
})

print("\n=== MASTER DATASET DI PROGETTO (MONITORAGGIO IMPIANTO ACQUE) ===")
display(master_progetto_df)

---

## 4: Architettura della Dashboard Streamlit (Integrazione UI)

Lo script seguente mostra la struttura logica dell'applicazione Streamlit (`app.py`) che unisce la visualizzazione della cartella clinica del paziente e l'analisi storica dei referti gestiti dall'IA.

In [ ]:
# Codice di struttura della Web App Streamlit per il Monitoraggio Acque di Impianto
"""
import streamlit as st
import pandas as pd

st.set_page_config(page_title="Monitoraggio Impianto Acque", layout="wide")

st.title("💧 Monitoraggio Acque di Impianto & Local AI Dashboard")
st.sidebar.header("Pannello di Controllo")

# Caricamento dataset pulito dei referti d'impianto
@st.cache_data
def carica_dati():
    return pd.read_csv("referti_acque_clean.csv")

df = carica_dati()

# Widget interattivi per il filtraggio dei dati dell'impianto
punto_selezionato = st.sidebar.selectbox("Seleziona Punto di Prelievo", ["Tutti"] + list(df['punto_prelievo'].unique()))
parametro_selezionato = st.sidebar.selectbox("Seleziona Parametro", ["Tutti", "pH", "Nitrati", "Conducibilità", "COD"])

# Filtro rapido per evidenziare le criticità/anomalia dell'impianto
solo_non_conformi = st.sidebar.checkbox("Mostra solo anomalie (Non Conforme)")
"""

print("Struttura della Web App Streamlit adattata all'analisi acque di impianto pronta per il deployment locale.")

## 5: Approfondimenti e Messa in Produzione

Nella pratica industriale e nella gestione avanzata degli impianti di trattamento o depurazione delle acque, un sistema basato su LLM locali e referti tecnici può evolvere verso architetture software di livello superiore. 

L'integrazione di tecnologie avanzate consente di trasformare un semplice strumento di parsing in un vero e proprio assistente decisionale per il plant manager e il responsabile della conformità ambientale.

---

### Architetture Avanzate per il Monitoraggio delle Acque

* **Vector Database e RAG (Retrieval-Augmented Generation):** 
  * *Come funziona:* Tutti i referti storici di laboratorio (convertiti in embedding vettoriali) vengono archiviati in un Vector Database (es. ChromaDB o FAISS).
  * *Vantaggio pratico:* L'LLM locale (Ollama) non si limita a leggere il singolo documento corrente, ma può attingere a anni di dati storici dell'impianto per rispondere a domande complesse in linguaggio naturale.
* **Automazione dei flussi (Workflow Automation):** 
  * *Come funziona:* Configurazione di script demone in background (tramite librerie Python come `watchdog`) che monitorano una cartella di rete condivisa o una casella PEC aziendale.
  * *Vantaggio pratico:* Non appena il laboratorio esterno invia via email o deposita un nuovo file PDF di analisi, la pipeline si attiva in tempo reale senza intervento umano, estrae i dati, calcola la conformità e aggiorna il database master e la dashboard Streamlit.

---

### Esempio Concreto: Automazione del flusso e Interrogazione RAG in Impianto

Immaginiamo di voler implementare un demone di sistema in Python che intercetta automaticamente l'arrivo di un nuovo referto di analisi delle acque nella cartella dell'impianto e, in caso di non conformità, attiva un'interrogazione semantica (RAG) per suggerire azioni correttive basate sullo storico.


In [ ]:
import os
import time
import pandas as pd

# Simulazione di un demone di monitoraggio cartella (Event-Driven Automation)
def demone_monitoraggio_acque(cartella_monitorata: str):
    """
    Simula un processo in background che controlla l'arrivo di nuovi PDF 
    e avvia la pipeline end-to-end in modo totalmente autonomo.
    """
    print(f"Avvio del demone di ascolto sulla cartella impianto: '{cartella_monitorata}'...")
    
    # Simulazione rilevamento nuovo file PDF
    nuovo_file_rilevato = "report_laboratorio_linea_2026_07.pdf"
    print(f"[EVENTO] Rilevato nuovo referto: {nuovo_file_rilevato}")
    
    # 1. Esecuzione automatica della pipeline di estrazione e verifica limiti
    # (Richiama la pipeline sviluppata nei moduli precedenti)
    print("Esecuzione parsing OCR/Text e validazione Pydantic con Ollama...")
    
    # Esito simulato dell'estrazione automatica
    esito_analisi = {
        "punto_prelievo": "Scarico Principale Linea 2",
        "parametro": "COD (Domanda Chimica di Ossigeno)",
        "valore": 175.0,
        "limite_massimo": 160.0,
        "conforme_globale": False
    }
    
    # 2. Gestione automatica dell'allerta e attivazione RAG in caso di anomalia
    if not esito_analisi["conforme_globale"]:
        print("\n⚠️ [ALLARME IMPIANTO] Rilevata Non Conformità!")
        print(f"Parametro critico: {esito_analisi['parametro']} = {esito_analisi['valore']} (Limite max: {esito_analis_max = esito_analisi['limite_massimo'] if 'limite_massimo' in esito_analisi else 160.0})")
        
        # Interrogazione RAG sullo storico per supporto decisionale
        print("\n[RAG Engine] Interrogazione dello storico dei trattamenti passati...")
        suggerimento_llm = (
            "Analisi dello storico: Negli ultimi 12 mesi, superamenti analoghi del COD "
            "sulla Linea 2 sono stati risolti aumentando del 15% l'apporto di ossigeno "
            "nella vasca di ossidazione biologica e verificando il dosaggio del polielettrolita."
        )
        print(f"Suggerimento generato dall'IA locale:\n> {suggerimento_llm}")
    else:
        print("\n[OK] Referto conforme. Nessuna azione richiesta.")

# Esecuzione della simulazione pratica
demone_monitoraggio_acque("/var/log/impianto/referti_ingressi")

---

## Esercitazione Pratica & Live Test

### Task finale:
Simula il caricamento di un nuovo referto PDF sconosciuto tramite l'interfaccia, esegui il parsing con il modulo Ollama/Pydantic, fondi il risultato nel Master Dataset e verifica che l'anomalia venga intercettata correttamente dai filtri della dashboard.

In [ ]:
# === SIMULAZIONE LIVE TEST & TROUBLESHOOTING IMPIANTO ACQUE ===
def simula_live_test():
    nuovo_file = "referto_nuovo_impianto_test.pdf"
    print(f"Simulazione caricamento e analisi in tempo reale del file d'impianto: {nuovo_file}")
    
    # Esecuzione pipeline e simulazione nuovo record con verifica dei limiti per le acque
    nuovo_record = pd.DataFrame({
        'punto_prelievo': ['Sezione Trattamento B'],
        'data_campionamento': ['2026-07-31'],
        'laboratorio': ['EcoLab Esterno'],
        'parametro': ['Torbidità'],
        'valore': [1.8],
        'unita': ['NTU'],
        'limite_minimo': [None],
        'limite_massimo': [1.0], # Il valore supera il limite massimo consentito (1.8 > 1.0)
        'conforme_parametro': [False],
        'conforme_globale': [False]
    })
    
    global master_progetto_df
    master_progetto_df = pd.concat([master_progetto_df, nuovo_record], ignore_index=True)
    
    print("\n[OK] Referto d'impianto elaborato con successo dall'IA locale!")
    print(f"Totale record aggiornati nel sistema: {len(master_progetto_df)}")
    print("\nControllo non conformità d'impianto aggiornato in tempo reale:")
    
    # Filtro delle non conformità basato sul flag dei singoli parametri
    non_conformi = master_progetto_df[master_progetto_df['conforme_parametro'] == False]
    display(non_conformi[['data_campionamento', 'punto_prelievo', 'laboratorio', 'parametro', 'valore', 'limite_massimo', 'conforme_parametro']])

simula_live_test()